<a href="https://colab.research.google.com/github/Mr-PeterMaged/flyrank/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mr-PeterMaged/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (Lane 2).**

I'm picking Lane 2 over the others because the starter dataset already shows a large, well-populated
candidate pool for it: **54.2%** of the 30,000 rows carry `trend_direction == "down"`, and **9,759 rows
(32.5%)** match a "visible but under-clicking" pattern (`impressions_90d >= 500`, position 1-20,
`ctr < 0.5`, computed in section 3 below). That's enough volume to build and honestly validate a ranked
review queue, unlike the AI-referral freestyle direction, where only **6.4%** of rows have any AI
sessions at all — too sparse to support supervised work. Lane 3 (clustering) is interesting but doesn't
map as directly onto a single recurring decision ("what do I review next"), and Lane 1 (signal analysis)
is closer to a step inside Lane 2 than a full project on its own.

The starter pipeline (`scripts/01`-`05`) already proves the shape of this workflow works on this data:
a random forest trained on the same 30,000 rows reaches **precision@50 = 0.740** versus **0.240** for the
hand-written rule baseline (`outputs/model_report.md`) — roughly 3x more true positives in the top 50
candidates. That result also comes with a real weakness I want to fix, not repeat: the starter label
(`is_declining_label = trend_direction == "down"`) is a same-window proxy, not a future outcome. My
capstone plan is to keep Lane 2's shape (ranked queue with reason codes) but move the label to a
future-window definition — prior 90 days of features predicting a next-30-day decline/recovery — using
the warehouse release once I get there.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Out of a client's content inventory, which pages should a reviewer with limited
capacity look at first for refresh, metadata, or engagement work?

- **Unit of analysis:** one content page (`content_id`, scoped within its `client_id`), scored from its
  trailing ~90-day window of search and engagement signals.
- **Decision this improves:** how a content/SEO reviewer spends a fixed weekly review budget across a
  much larger inventory than they can fully audit by hand.
- **Output:** a ranked review queue per client — top-N pages, each with a score and one or more reason
  codes (e.g. `stale_visible_page`, `declining_with_demand`, `low_ctr_visible_page`) explaining *why* it
  surfaced, so the reviewer can sanity-check the ranking, not just trust it blindly.
- **Who acts, and what they do:** a content strategist/editor reads the queue and picks an action per
  page — refresh the content, rewrite title/metadata, protect (leave alone), or keep monitoring. The
  model never takes the action itself; it only orders the reviewer's attention.
- **Cost of a wrong call:**
  - *False positive* (a fine page gets flagged): wastes a reviewer's scarce hours — the real cost here,
    since review capacity, not data, is the bottleneck.
  - *False negative* (a genuinely declining page is never surfaced): it keeps losing visibility/clicks
    unnoticed until someone stumbles on it — a slower, compounding cost.
  - Because capacity is fixed, **precision@K** (are the top K pages actually worth a look) matters more
    than overall accuracy across the whole inventory.
- **Why data/ML helps:** a single hand-written rule already exists here and it isn't enough — the
  starter baseline rules only catch **12 of the top 50** candidates that later matter
  (`precision@50 = 0.240`), while a learned score catches **37 of 50** (`precision@50 = 0.740`) on the
  same data. Real underperformance shows up as several signals moving together (position, CTR,
  freshness, trend, engagement) in ways that shift per client and content type — too tangled for one
  if-statement, but a pattern a model can pick up and a reviewer can still audit through reason codes.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

Loading the starter CSV and pulling a few numbers that back the lane choice above: how much of the
inventory already looks like a Lane 2 candidate, and how much headroom the starter pipeline's own
random forest showed over its rule baseline.

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
n = len(df)
print(f"rows: {n}, clients: {df['client_id'].nunique()}")

# 1) how much of the inventory is already flagged declining (the starter proxy label)
declining_rate = (df["trend_direction"] == "down").mean()
print(f"share of rows with trend_direction == 'down': {declining_rate:.1%}")

# 2) how much of the inventory looks like a "visible but under-clicking" Lane 2 candidate
low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)
print(f"'low_ctr_visible_page' candidates: {low_ctr_visible.sum()} rows ({low_ctr_visible.mean():.1%})")

# 3) how many rows match at least one starter reason code -> realistic size of a review queue
stale_visible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
declining_with_demand = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
page_one_decay = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
any_candidate = stale_visible | declining_with_demand | low_ctr_visible | page_one_decay
print(f"rows matching >=1 Lane 2 reason code: {any_candidate.sum()} ({any_candidate.mean():.1%})")

# 4) evidence a learned score beats the rule baseline on this same data (outputs/model_report.md)
print("\nFrom outputs/model_report.md (already-run starter pipeline, client-holdout validation):")
print("  rule baseline  precision@50 = 0.240 (12 / top 50 correct)")
print("  random forest  precision@50 = 0.740 (37 / top 50 correct)")

rows: 30000, clients: 32
share of rows with trend_direction == 'down': 54.2%
'low_ctr_visible_page' candidates: 9759 rows (32.5%)
rows matching >=1 Lane 2 reason code: 19624 (65.4%)

From outputs/model_report.md (already-run starter pipeline, client-holdout validation):
  rule baseline  precision@50 = 0.240 (12 / top 50 correct)
  random forest  precision@50 = 0.740 (37 / top 50 correct)


## 4. Careful words: what I can and can't claim

**Can claim, with evidence:**
- *Observed* associations — e.g. "in this snapshot, pages matching `low_ctr_visible_page` or
  `declining_with_demand` make up a specific, measurable share of the inventory."
- *Directional* ranking quality — "a learned score's top-K queue contains more truly-flagged pages than
  the rule baseline's top-K, measured by precision@K on a client-holdout test set" (already true of the
  starter pipeline: 0.740 vs 0.240 precision@50).
- *Decision-support* value — "this ranking is a reasonable starting point for a capacity-limited
  reviewer," not a verdict on any single page.

**Cannot claim:**
- That refreshing a flagged page *causes* recovery — that needs an experiment (e.g. an actual A/B test
  on refreshes), which this observational data cannot provide.
- Anything about Google's ranking algorithm — every signal here is an observed search/engagement
  outcome, not the algorithm's internals.
- That the starter label (`is_declining_label = trend_direction == "down"`) is the "true" target — it is
  a same-window proxy derived from `trend_pct`, not a future outcome. Once I build a stronger,
  future-window label, `trend_direction`/`trend_pct` become something to compare against, never a
  feature (the label trap noted in the data skill).
- Anything at the level of an individual identifiable client or page publicly — `client_id`/`content_id`
  are pseudonyms for grouping only; no client names, URLs, or raw queries appear anywhere in this work.

In [4]:
# Sanity check backing section 4: confirm no raw/private fields ship in this dataset,
# and that the id columns are pseudonyms (not human-readable), before I claim anything is public-safe.
unsafe_patterns = ["url", "domain", "title", "query", "keyword", "name", "email"]
unsafe_cols = [c for c in df.columns if any(p in c.lower() for p in unsafe_patterns)]
print("columns matching unsafe-field patterns:", unsafe_cols)
print("content_id sample (pseudonymized):", df["content_id"].iloc[0])
print("client_id sample (pseudonymized):", df["client_id"].iloc[0])

columns matching unsafe-field patterns: []
content_id sample (pseudonymized): content_304f48230142
client_id sample (pseudonymized): client_f369cb89fc


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.